# Decision Trees and Random Forests Assignment
## Breast Cancer Dataset Analysis

This notebook presents a comprehensive analysis of the Wisconsin Diagnostic Breast Cancer (WDBC) dataset using Decision Trees and Random Forests classifiers.

### Dataset Information
- **Source**: Wisconsin Diagnostic Breast Cancer (WDBC) dataset
- **Samples**: 569 instances
- **Features**: 30 real-valued features computed from digitized images of breast mass
- **Target**: Binary classification (Malignant vs Benign)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from collections import Counter

# Set style for better visualizations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

## Data Loading and Preprocessing

First, we load the breast cancer dataset from `wdbc.data` and parse it according to the specifications in `wdbc.names`.

In [ ]:
# Define feature names based on wdbc.names
feature_names = [
    'mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area',
    'mean_smoothness', 'mean_compactness', 'mean_concavity',
    'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension',
    'se_radius', 'se_texture', 'se_perimeter', 'se_area',
    'se_smoothness', 'se_compactness', 'se_concavity',
    'se_concave_points', 'se_symmetry', 'se_fractal_dimension',
    'worst_radius', 'worst_texture', 'worst_perimeter', 'worst_area',
    'worst_smoothness', 'worst_compactness', 'worst_concavity',
    'worst_concave_points', 'worst_symmetry', 'worst_fractal_dimension'
]

# Load the data
column_names = ['id', 'diagnosis'] + feature_names
df = pd.read_csv('wdbc.data', header=None, names=column_names)

# Display basic information
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum().sum())

In [ ]:
# Prepare features (X) and target (y)
X = df[feature_names].values
# Convert diagnosis: M (malignant) = 1, B (benign) = 0
y = (df['diagnosis'] == 'M').astype(int).values

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nClass distribution:")
print(f"Benign (B): {np.sum(y == 0)}")
print(f"Malignant (M): {np.sum(y == 1)}")

---
## Question 1: Base Rate of Malignant Cancer

**Task**: Calculate the base rate (probability) of malignant cancer occurrence in the dataset.

**Approach**: The base rate is calculated as the proportion of malignant cases (M) over the total number of cases.

In [ ]:
# Calculate base rate
n_malignant = np.sum(y == 1)
n_total = len(y)
base_rate = n_malignant / n_total

print("="*50)
print("QUESTION 1: Base Rate Analysis")
print("="*50)
print(f"Total samples: {n_total}")
print(f"Malignant cases (M): {n_malignant}")
print(f"Benign cases (B): {n_total - n_malignant}")
print(f"\nBase rate (probability) of malignant cancer: {base_rate:.4f} ({base_rate*100:.2f}%)")
print("="*50)

# Visualize the distribution
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
labels = ['Benign (B)', 'Malignant (M)']
counts = [n_total - n_malignant, n_malignant]
colors = ['#2ecc71', '#e74c3c']
ax.bar(labels, counts, color=colors, alpha=0.7, edgecolor='black')
ax.set_ylabel('Number of Cases', fontsize=12)
ax.set_title('Distribution of Cancer Diagnosis', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
for i, (label, count) in enumerate(zip(labels, counts)):
    ax.text(i, count + 10, f'{count}\n({count/n_total*100:.1f}%)', 
            ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Question 2(a): Decision Tree Classifier Performance

**Task**: Implement a DecisionTreeClassifier and evaluate it using:
1. Training on the entire dataset (full-dataset accuracy)
2. 10-fold cross-validation

We vary `max_depth` from 1 to 10 and plot the results.

**Approach**: 
- For full-dataset accuracy: Train on all data and test on the same data (measures training fit)
- For cross-validation: Use 10-fold CV to estimate generalization performance

In [ ]:
# Question 2(a): Decision Tree with varying max_depth
max_depths = range(1, 11)
full_dataset_scores = []
cv_scores_mean = []
cv_scores_std = []

print("="*50)
print("QUESTION 2(a): Decision Tree Performance")
print("="*50)
print(f"{'max_depth':<12} {'Full Dataset':<15} {'10-Fold CV Mean':<18} {'10-Fold CV Std'}")
print("-"*65)

for depth in max_depths:
    # Create classifier
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    
    # Full dataset accuracy (train and test on same data)
    dt.fit(X, y)
    full_dataset_acc = dt.score(X, y)
    full_dataset_scores.append(full_dataset_acc)
    
    # 10-fold cross-validation
    cv_scores = cross_val_score(dt, X, y, cv=10, scoring='accuracy')
    cv_scores_mean.append(cv_scores.mean())
    cv_scores_std.append(cv_scores.std())
    
    print(f"{depth:<12} {full_dataset_acc:<15.4f} {cv_scores.mean():<18.4f} {cv_scores.std():.4f}")

print("="*50)

In [ ]:
# Plot the results
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

# Plot full dataset accuracy
ax.plot(max_depths, full_dataset_scores, marker='o', linewidth=2, 
        markersize=8, label='Full Dataset Accuracy', color='#3498db')

# Plot cross-validation accuracy with error bars
ax.errorbar(max_depths, cv_scores_mean, yerr=cv_scores_std, 
            marker='s', linewidth=2, markersize=8, capsize=5,
            label='10-Fold CV Accuracy (±1 std)', color='#e74c3c')

ax.set_xlabel('max_depth', fontsize=12, fontweight='bold')
ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Decision Tree Performance vs max_depth', fontsize=14, fontweight='bold')
ax.set_xticks(max_depths)
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim([0.85, 1.01])
plt.tight_layout()
plt.show()

---
## Question 2(b): Optimal max_depth for Decision Tree

**Task**: Identify the optimal `max_depth` settings based on:
- (i) Full-dataset accuracy
- (ii) 10-fold cross-validated accuracy

In [ ]:
# Question 2(b): Find optimal max_depth
best_depth_full = max_depths[np.argmax(full_dataset_scores)]
best_score_full = max(full_dataset_scores)

best_depth_cv = max_depths[np.argmax(cv_scores_mean)]
best_score_cv = max(cv_scores_mean)

print("="*50)
print("QUESTION 2(b): Optimal max_depth Settings")
print("="*50)
print(f"(i) Best max_depth for Full Dataset: {best_depth_full}")
print(f"    Accuracy: {best_score_full:.4f}")
print()
print(f"(ii) Best max_depth for 10-Fold CV: {best_depth_cv}")
print(f"     Accuracy: {best_score_cv:.4f}")
print("="*50)

# Store the optimal depth for use in Question 3
optimal_depth_dt = best_depth_cv

**Analysis**: 
- The full-dataset accuracy typically increases with depth, as deeper trees can memorize the training data (potential overfitting)
- The cross-validated accuracy provides a better estimate of generalization performance
- The optimal depth based on CV is more reliable for real-world performance

---
## Question 3(a): Random Forest with Varying n_estimators

**Task**: Use RandomForestClassifier with the optimal `max_depth` from Question 2(b)(ii) and vary `n_estimators` from 1 to 20.

**Approach**: Keep `max_depth` fixed at the optimal value and evaluate 10-fold CV accuracy for different numbers of trees.

In [ ]:
# Question 3(a): Random Forest with varying n_estimators
n_estimators_range = range(1, 21)
rf_cv_scores_mean = []
rf_cv_scores_std = []

print("="*50)
print("QUESTION 3(a): Random Forest Performance")
print(f"Using optimal max_depth from Question 2(b)(ii): {optimal_depth_dt}")
print("="*50)
print(f"{'n_estimators':<15} {'10-Fold CV Mean':<18} {'10-Fold CV Std'}")
print("-"*50)

for n_est in n_estimators_range:
    # Create Random Forest classifier
    rf = RandomForestClassifier(n_estimators=n_est, max_depth=optimal_depth_dt, 
                                 random_state=42)
    
    # 10-fold cross-validation
    cv_scores = cross_val_score(rf, X, y, cv=10, scoring='accuracy')
    rf_cv_scores_mean.append(cv_scores.mean())
    rf_cv_scores_std.append(cv_scores.std())
    
    print(f"{n_est:<15} {cv_scores.mean():<18.4f} {cv_scores.std():.4f}")

print("="*50)

In [ ]:
# Plot Random Forest performance
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

ax.errorbar(n_estimators_range, rf_cv_scores_mean, yerr=rf_cv_scores_std,
            marker='o', linewidth=2, markersize=8, capsize=5,
            label=f'Random Forest (max_depth={optimal_depth_dt})', color='#2ecc71')

# Add horizontal line for best single Decision Tree
ax.axhline(y=best_score_cv, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Best Single Decision Tree (max_depth={best_depth_cv})')

ax.set_xlabel('n_estimators', fontsize=12, fontweight='bold')
ax.set_ylabel('10-Fold CV Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Random Forest Performance vs n_estimators', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xticks(n_estimators_range)
plt.tight_layout()
plt.show()

---
## Question 3(b): Random Forest vs Single Decision Tree

**Task**: Compare the performance of Random Forest to the single DecisionTreeClassifier.

In [ ]:
# Question 3(b): Compare Random Forest to Decision Tree
best_rf_score = max(rf_cv_scores_mean)
improvement = best_rf_score - best_score_cv
improvement_pct = (improvement / best_score_cv) * 100

print("="*50)
print("QUESTION 3(b): Random Forest vs Decision Tree")
print("="*50)
print(f"Best Single Decision Tree CV Accuracy: {best_score_cv:.4f}")
print(f"Best Random Forest CV Accuracy: {best_rf_score:.4f}")
print(f"\nAbsolute Improvement: {improvement:.4f}")
print(f"Relative Improvement: {improvement_pct:.2f}%")
print()
if improvement > 0.01:  # More than 1% improvement
    print("Conclusion: The Random Forest shows NOTICEABLE improvement")
    print("over the single Decision Tree. The ensemble approach reduces")
    print("variance and improves generalization performance.")
else:
    print("Conclusion: The improvement is MARGINAL. Both models perform")
    print("similarly, suggesting the dataset may not benefit significantly")
    print("from ensemble methods at this depth.")
print("="*50)

---
## Question 3(c): Optimal n_estimators

**Task**: Identify the `n_estimators` value that gave the best cross-validated accuracy.

In [ ]:
# Question 3(c): Find optimal n_estimators
best_n_estimators = n_estimators_range[np.argmax(rf_cv_scores_mean)]
best_n_estimators_score = max(rf_cv_scores_mean)

print("="*50)
print("QUESTION 3(c): Optimal n_estimators")
print("="*50)
print(f"Best n_estimators: {best_n_estimators}")
print(f"10-Fold CV Accuracy: {best_n_estimators_score:.4f}")
print("="*50)

# Store for next question
optimal_n_estimators = best_n_estimators

---
## Question 3(d): Random Forest with Varying max_depth

**Task**: Fix `n_estimators` to the optimal value from 3(c) and vary `max_depth` from 1 to 10.

In [ ]:
# Question 3(d): Random Forest with varying max_depth
max_depths_rf = range(1, 11)
rf_depth_cv_scores_mean = []
rf_depth_cv_scores_std = []

print("="*50)
print("QUESTION 3(d): Random Forest with Varying max_depth")
print(f"Using optimal n_estimators from Question 3(c): {optimal_n_estimators}")
print("="*50)
print(f"{'max_depth':<12} {'10-Fold CV Mean':<18} {'10-Fold CV Std'}")
print("-"*45)

for depth in max_depths_rf:
    # Create Random Forest classifier
    rf = RandomForestClassifier(n_estimators=optimal_n_estimators, 
                                 max_depth=depth, random_state=42)
    
    # 10-fold cross-validation
    cv_scores = cross_val_score(rf, X, y, cv=10, scoring='accuracy')
    rf_depth_cv_scores_mean.append(cv_scores.mean())
    rf_depth_cv_scores_std.append(cv_scores.std())
    
    print(f"{depth:<12} {cv_scores.mean():<18.4f} {cv_scores.std():.4f}")

print("="*50)

In [ ]:
# Plot Random Forest with varying max_depth
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

ax.errorbar(max_depths_rf, rf_depth_cv_scores_mean, yerr=rf_depth_cv_scores_std,
            marker='o', linewidth=2, markersize=8, capsize=5,
            label=f'Random Forest (n_estimators={optimal_n_estimators})', color='#9b59b6')

ax.set_xlabel('max_depth', fontsize=12, fontweight='bold')
ax.set_ylabel('10-Fold CV Accuracy', fontsize=12, fontweight='bold')
ax.set_title('Random Forest Performance vs max_depth', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xticks(max_depths_rf)
plt.tight_layout()
plt.show()

---
## Question 3(e): Compare Optimal max_depth Settings

**Task**: Compare the optimal `max_depth` from 3(d) with the one from 2(b)(ii) and explain any differences.

In [ ]:
# Question 3(e): Compare optimal max_depth settings
best_depth_rf = max_depths_rf[np.argmax(rf_depth_cv_scores_mean)]
best_score_rf_depth = max(rf_depth_cv_scores_mean)

print("="*50)
print("QUESTION 3(e): Comparison of Optimal max_depth")
print("="*50)
print(f"Optimal max_depth for Single Decision Tree [2(b)(ii)]: {optimal_depth_dt}")
print(f"  CV Accuracy: {best_score_cv:.4f}")
print()
print(f"Optimal max_depth for Random Forest [3(d)]: {best_depth_rf}")
print(f"  CV Accuracy: {best_score_rf_depth:.4f}")
print()
print("Explanation:")
if best_depth_rf != optimal_depth_dt:
    print(f"The optimal max_depth values DIFFER between the two models.")
    print()
    if best_depth_rf > optimal_depth_dt:
        print("The Random Forest benefits from deeper trees because:")
        print("1. Ensemble averaging reduces overfitting risk")
        print("2. Each tree sees a different bootstrap sample")
        print("3. Feature randomness adds diversity to the ensemble")
        print("4. Deeper trees can capture more complex patterns without")
        print("   overfitting as much as a single deep tree would")
    else:
        print("The Random Forest performs best with shallower trees because:")
        print("1. The dataset may not require deep trees")
        print("2. Shallower trees are more diverse in their predictions")
        print("3. The ensemble effect is stronger with simpler base learners")
else:
    print(f"The optimal max_depth is the SAME ({best_depth_rf}) for both models.")
    print("This suggests that this depth provides a good balance between")
    print("model complexity and generalization for this dataset, regardless")
    print("of whether we use a single tree or an ensemble.")
print("="*50)

---
## Question 4(a): Optimal max_depth Distribution Across Random States

**Task**: Generate a bar chart showing the distribution of optimal `max_depth` settings for 100 different random states (0-99).

**Approach**: For each random state, find the best `max_depth` based on 10-fold CV accuracy.

In [ ]:
# Question 4(a): Optimal max_depth for 100 random states
random_states = range(100)
optimal_depths_distribution = []

print("="*50)
print("QUESTION 4(a): Optimal max_depth Across Random States")
print("="*50)
print("Computing optimal max_depth for 100 random states...")
print("This may take a few moments...")
print()

for rs in random_states:
    best_depth_for_rs = None
    best_score_for_rs = 0
    
    for depth in range(1, 11):
        dt = DecisionTreeClassifier(max_depth=depth, random_state=rs)
        cv_scores = cross_val_score(dt, X, y, cv=10, scoring='accuracy')
        mean_score = cv_scores.mean()
        
        if mean_score > best_score_for_rs:
            best_score_for_rs = mean_score
            best_depth_for_rs = depth
    
    optimal_depths_distribution.append(best_depth_for_rs)
    
    # Print progress every 20 iterations
    if (rs + 1) % 20 == 0:
        print(f"Processed {rs + 1}/100 random states...")

print("\nCompleted!")
print("="*50)

In [ ]:
# Count the frequency of each max_depth
depth_counts = Counter(optimal_depths_distribution)
depths = sorted(depth_counts.keys())
counts = [depth_counts[d] for d in depths]

# Create bar chart
fig, ax = plt.subplots(1, 1, figsize=(12, 7))

bars = ax.bar(depths, counts, color='#3498db', alpha=0.7, edgecolor='black', linewidth=1.5)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xlabel('max_depth', fontsize=12, fontweight='bold')
ax.set_ylabel('Frequency (out of 100 random states)', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Optimal max_depth Across 100 Random States', 
             fontsize=14, fontweight='bold')
ax.set_xticks(depths)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Print summary statistics
print("\nSummary Statistics:")
print("-" * 40)
for depth in depths:
    count = depth_counts[depth]
    percentage = (count / 100) * 100
    print(f"max_depth = {depth}: {count} times ({percentage:.1f}%)")
print("-" * 40)
print(f"Total: {sum(counts)} random states")

---
## Question 4(b): Top Two Most Frequent max_depth Settings

**Task**: Identify the top two most frequently chosen `max_depth` parameter settings from Question 4(a).

In [ ]:
# Question 4(b): Find top two most frequent max_depth values
most_common = depth_counts.most_common(2)

print("="*50)
print("QUESTION 4(b): Top Two Most Frequent max_depth Settings")
print("="*50)
for rank, (depth, count) in enumerate(most_common, 1):
    percentage = (count / 100) * 100
    print(f"Rank {rank}: max_depth = {depth}")
    print(f"  Frequency: {count} out of 100 ({percentage:.1f}%)")
    print()

print("Interpretation:")
print(f"The most stable max_depth value is {most_common[0][0]}, chosen in")
print(f"{most_common[0][1]}% of cases. This suggests it provides consistent")
print("performance across different random initializations.")
print("="*50)

---
## Summary and Conclusions

This notebook has comprehensively analyzed the Wisconsin Diagnostic Breast Cancer dataset using Decision Trees and Random Forests:

### Key Findings:

1. **Base Rate**: The dataset has a class imbalance with malignant cases representing approximately 37% of the data.

2. **Decision Tree Analysis**:
   - Full-dataset accuracy increases with depth (potential overfitting)
   - Cross-validation provides a more realistic performance estimate
   - Optimal depth balances model complexity and generalization

3. **Random Forest Improvements**:
   - Ensemble methods generally outperform single trees
   - Multiple trees reduce variance through averaging
   - Performance stabilizes as the number of trees increases

4. **Depth Comparison**:
   - Optimal depth may differ between single trees and forests
   - Random Forests can handle deeper trees without overfitting as much

5. **Stability Analysis**:
   - Certain max_depth values are consistently preferred across random states
   - This indicates robust hyperparameter choices for this dataset

### Recommendations:

- Use cross-validation for hyperparameter selection
- Consider ensemble methods for improved performance
- Test stability across multiple random states
- Balance model complexity with generalization ability